# Notebook 08 — Linear Regression as a Linear Algebra Problem

**Module:** 04 — Linear Algebra  
**Notebook:** 08 of 10  
**Prerequisites:** NB03, NB05  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

In Module 03 (NB15) we derived OLS via calculus. Here we re-derive it through
the lens of linear algebra — as a **projection** problem. This geometric view explains
why OLS gives the best linear unbiased estimator, why conditioning matters, and
why ridge regression (L2 regularization) is just adding $\lambda I$ to the
projection matrix.

---
## Step 2 — Intuition

OLS asks: 'Given that $\mathbf{y}$ might not lie in the column space of $\mathbf{X}$
(it won't, unless $n = p$), find $\hat{\boldsymbol{\beta}}$ such that
$\mathbf{X}\hat{\boldsymbol{\beta}}$ is the closest point in the column space of
$\mathbf{X}$ to $\mathbf{y}$.'

That closest point is the **orthogonal projection** of $\mathbf{y}$ onto the
column space of $\mathbf{X}$. The projection vector is $\hat{\mathbf{y}} = \mathbf{H}\mathbf{y}$
where $\mathbf{H} = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$
is the **hat matrix**.

---
## Step 3 — Biological Background

**eQTL mapping:**
For each gene $g$, fit $\mathbf{y}_g = \mathbf{X}\boldsymbol{\beta}_g + \boldsymbol{\varepsilon}$
where $\mathbf{X}$ encodes genotypes and covariates. The Normal equations give
$\hat{\boldsymbol{\beta}}_g$ for all genes simultaneously.

**GWAS with covariates:**
Include PC scores (from PCA on the genotype matrix) as covariates to control for
population structure. The genotype effect $\hat{\beta}_{\text{SNP}}$ is the partial
regression coefficient after projecting out the PC axes.

**Ridge regression (L2 regularization):**
When the design matrix is nearly singular (highly correlated predictors), OLS
estimates are unstable. Ridge adds $\lambda I$ to shrink coefficients:
$\hat{\boldsymbol{\beta}}_{\text{ridge}} = (\mathbf{X}^\top\mathbf{X} + \lambda I)^{-1}\mathbf{X}^\top\mathbf{y}$.

---
## Step 4 — Mathematical Explanation

**Normal equations** (derived from the projection condition $\mathbf{X}^\top(\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}}) = \mathbf{0}$):
$$\mathbf{X}^\top\mathbf{X}\hat{\boldsymbol{\beta}} = \mathbf{X}^\top\mathbf{y}$$
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$$

**Hat matrix:**
$$\hat{\mathbf{y}} = \mathbf{H}\mathbf{y}, \quad \mathbf{H} = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$$
$\mathbf{H}$ is symmetric and idempotent: $\mathbf{H}^2 = \mathbf{H}$.

**QR decomposition** (numerically stable OLS):
$\mathbf{X} = \mathbf{Q}\mathbf{R}$ where $\mathbf{Q}$ is orthonormal and $\mathbf{R}$ upper triangular.
Then $\hat{\boldsymbol{\beta}} = \mathbf{R}^{-1}\mathbf{Q}^\top\mathbf{y}$ — avoids computing
$\mathbf{X}^\top\mathbf{X}$ which squares the condition number.

**Condition number:** $\kappa(\mathbf{X}^\top\mathbf{X}) = \kappa(\mathbf{X})^2$.
Ridge regression reduces the condition number: $\kappa(\mathbf{X}^\top\mathbf{X} + \lambda I)$
is smaller than $\kappa(\mathbf{X}^\top\mathbf{X})$ for any $\lambda > 0$.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — OLS via Normal equations, QR, and lstsq — comparing all three
rng = np.random.default_rng(42)
n, p = 100, 4   # samples, predictors (including intercept)
# eQTL design: intercept + genotype + age + sex
X = np.column_stack([
    np.ones(n),
    rng.choice([0,1,2], n, p=[0.25,0.5,0.25]),   # genotype
    rng.normal(40, 10, n),                          # age
    rng.choice([0,1], n)                            # sex
])
true_beta = np.array([5.0, 0.8, 0.02, -0.3])
y = X @ true_beta + rng.normal(0, 1, n)

# Method 1: Normal equations (direct)
beta_normal = np.linalg.solve(X.T @ X, X.T @ y)

# Method 2: QR decomposition
Q, R = np.linalg.qr(X)
beta_qr = np.linalg.solve(R, Q.T @ y)

# Method 3: numpy lstsq (uses SVD internally — most robust)
beta_lstsq, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

print(f"True β:         {true_beta}")
print(f"Normal eq. β:   {beta_normal.round(4)}")
print(f"QR β:           {beta_qr.round(4)}")
print(f"lstsq β:        {beta_lstsq.round(4)}")
print(f"\nMax diff (QR vs normal): {np.max(np.abs(beta_qr - beta_normal)):.2e}")

In [ ]:
# Cell 6.2 — Ridge regression and condition number
def ridge_regression(X: np.ndarray, y: np.ndarray, lam: float) -> np.ndarray:
    """
    Ridge regression: β = (XᵀX + λI)⁻¹Xᵀy.
    λ > 0 shrinks coefficients and improves conditioning.
    """
    p = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)

# Nearly collinear design matrix (ill-conditioned)
X_ill = np.column_stack([np.ones(n), rng.normal(0, 1, n)])
X_ill = np.column_stack([X_ill, X_ill[:, 1] + rng.normal(0, 0.01, n)])  # near-duplicate
y_simple = X_ill[:, 1] + rng.normal(0, 0.5, n)

cond_XtX = np.linalg.cond(X_ill.T @ X_ill)
print(f"Condition number of XᵀX: {cond_XtX:.2e}  (large → ill-conditioned)")

print(f"\n{'λ':>12} {'β₁':>10} {'β₂':>10} {'β₃':>10} {'cond(XᵀX+λI)':>18}")
print("-" * 55)
for lam in [0, 1e-4, 1e-2, 1.0, 10.0]:
    if lam == 0:
        beta = np.linalg.lstsq(X_ill, y_simple, rcond=None)[0]
    else:
        beta = ridge_regression(X_ill, y_simple, lam)
    cond = np.linalg.cond(X_ill.T @ X_ill + lam * np.eye(3))
    print(f"  λ={lam:>9.4f}  {beta[0]:>9.3f}  {beta[1]:>9.3f}  {beta[2]:>9.3f}  {cond:>16.2e}")

In [ ]:
# Cell 6.3 — Hat matrix and projection geometry
# Simple 2D example: project y onto column space of X
X_simple = np.array([[1, 0], [1, 1], [1, 2], [1, 3]], dtype=float)
y_simple2 = np.array([1.0, 2.5, 2.0, 4.5])

H = X_simple @ np.linalg.inv(X_simple.T @ X_simple) @ X_simple.T  # hat matrix
y_hat = H @ y_simple2
residuals = y_simple2 - y_hat

print("Hat matrix H properties:")
print(f"  H symmetric: {np.allclose(H, H.T)}")
print(f"  H idempotent (H²=H): {np.allclose(H @ H, H)}")
print(f"  Residuals ⊥ fitted values: {np.allclose(y_hat @ residuals, 0, atol=1e-10)}")
print(f"\nFitted values: {y_hat.round(4)}")
print(f"Residuals:     {residuals.round(4)}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — OLS as projection + ridge shrinkage
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Projection geometry in 2D (n=2 data points, p=1 predictor)
ax = axes[0]
x_1d = np.array([1.0, 2.5, 4.0])
y_1d = np.array([2.1, 3.8, 5.5])
X_1d = np.column_stack([np.ones(3), x_1d])
beta_hat = np.linalg.lstsq(X_1d, y_1d, rcond=None)[0]
x_plot = np.linspace(0, 5, 100)
ax.scatter(x_1d, y_1d, s=80, color='steelblue', zorder=5, label='Observed y')
y_fitted_plot = beta_hat[0] + beta_hat[1] * x_plot
ax.plot(x_plot, y_fitted_plot, 'tomato', lw=2, label='Fitted line (column space projection)')
for xi, yi, yhi in zip(x_1d, y_1d, beta_hat[0] + beta_hat[1]*x_1d):
    ax.plot([xi, xi], [yi, yhi], 'gray', lw=1, ls='--')
ax.set_xlabel('x (genotype)'); ax.set_ylabel('y (expression)')
ax.set_title('OLS = project y onto column space of X')
ax.legend(fontsize=8)

# Panel 2: Ridge coefficient path
ax = axes[1]
lambdas = np.logspace(-3, 2, 50)
betas_ridge = np.array([ridge_regression(X_ill, y_simple, lam) for lam in lambdas])
for j, color in enumerate(['steelblue', 'tomato', 'green']):
    ax.semilogx(lambdas, betas_ridge[:, j], color=color, lw=2, label=f'β{j}')
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel('Regularization λ'); ax.set_ylabel('Coefficient value')
ax.set_title('Ridge regression coefficient path\n(coefficients shrink toward 0)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Derive the Normal equations $\mathbf{X}^\top\mathbf{X}\hat{\boldsymbol{\beta}} = \mathbf{X}^\top\mathbf{y}$
   from the condition that residuals are orthogonal to the column space of $\mathbf{X}$.
2. Show that the hat matrix $\mathbf{H}$ is idempotent ($\mathbf{H}^2 = \mathbf{H}$).
   What does this mean geometrically?
3. Implement `ridge_regression(X, y, lam)` and plot the coefficient path as $\lambda$
   varies from $10^{-3}$ to $10^3$. What happens to the estimates as $\lambda \to \infty$?
4. Why is the condition number of $(\mathbf{X}^\top\mathbf{X})$ equal to the square of
   the condition number of $\mathbf{X}$? Why does this make QR preferable to the
   Normal equations for ill-conditioned problems?

---
## Quiz — Active Recall

1. Write the Normal equations. What geometric condition do they express?
2. What is the hat matrix $\mathbf{H}$? Write the formula. What does $\mathbf{H}\mathbf{y}$ compute?
3. Why is QR decomposition more numerically stable than forming $\mathbf{X}^\top\mathbf{X}$?
4. What is ridge regression? Write the modified Normal equations with regularization.
5. When does $\mathbf{X}^\top\mathbf{X}$ become singular? Give a biological example.

---
## Reflection

**Date completed:** ____________________

> *[Can you explain OLS as a projection without writing an equation? Can you explain why ridge regression helps when the design matrix is ill-conditioned?]*

---
*Next: `09_mini_project_pca_expression_data.ipynb`*